# PVNet training-loop smoke test: January 2024 data

This notebook runs **one training batch of two samples and one validation batch of one sample** through the real PVNet Lightning training module, using the January satellite, UKV and PV_Live stores already prepared for GSP 199. The two training samples are needed because PVNet’s fusion network uses BatchNorm, which cannot train on a batch of one. It checks dataset construction, batch shapes, model forward/backward, optimizer step and validation loss.

This is a pipeline check only. The model starts with random weights, and these three examples cannot measure forecast skill. It does not change the repository's main configs, download data, contact W&B, or save checkpoints.

The training examples are 2024-01-01 13:00 and 14:00 UTC; validation is 2024-01-04 13:00 UTC, after the satellite outage. These times were chosen because the saved UKV store supplies all five requested hourly frames for each. The split is by date to avoid leakage between train and validation.


## 1. Locate and check the existing data stores

In [10]:
from pathlib import Path
import os

# Support notebook kernels started either at PVNet/ or PVNet/Luke/.
REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/model/late_fusion.yaml").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the PVNet repository root from the notebook working directory.")
DATA_DIR = REPO_ROOT / "Luke/data/to_gitignore"
OUTPUT_DIR = DATA_DIR / "pvnet_training_smoke_test"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SATELLITE_STORE = DATA_DIR / "seviri_gsp199_jan1_week_sampler.zarr"
UKV_STORE = DATA_DIR / "ukv_gsp199_jan1_week_sampler_hcct_fixed.zarr"
PVLIVE_STORE = DATA_DIR / "pvlive_gsp199_jan1_week_sampler.zarr"
DATA_CONFIG_SOURCE = REPO_ROOT / "configs/datamodule/configuration/example_configuration.yaml"
MODEL_CONFIG_PATH = REPO_ROOT / "configs/model/late_fusion.yaml"

required = [SATELLITE_STORE, UKV_STORE, PVLIVE_STORE, DATA_CONFIG_SOURCE, MODEL_CONFIG_PATH]
missing = [p for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing smoke-test input(s):\n" + "\n".join(map(str, missing)))

print("Repository:", REPO_ROOT)
print("Smoke-test outputs:", OUTPUT_DIR)
for path in [SATELLITE_STORE, UKV_STORE, PVLIVE_STORE]:
    print(f"Found {path.name}")


Repository: /Users/mans5211/GitHub-repos/PVNet
Smoke-test outputs: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/pvnet_training_smoke_test
Found seviri_gsp199_jan1_week_sampler.zarr
Found ukv_gsp199_jan1_week_sampler_hcct_fixed.zarr
Found pvlive_gsp199_jan1_week_sampler.zarr


## 2. Make a temporary data config that matches the checked-in model

The January sampler stores include UKV only, so this cell removes ECMWF from the example input config and points to the local stores. It sets generation to −2 h…+3 h and UKV to −2 h…+2 h, matching `configs/model/late_fusion.yaml`. The original config files are left unchanged.


In [11]:
import yaml

with DATA_CONFIG_SOURCE.open() as f:
    data_config = yaml.safe_load(f)
input_data = data_config["input_data"]

input_data["nwp"] = {"ukv": input_data["nwp"]["ukv"]}
generation_cfg = input_data["generation"]
generation_cfg["zarr_path"] = str(PVLIVE_STORE.resolve())
generation_cfg["interval_start_minutes"] = -120
generation_cfg["interval_end_minutes"] = 180
generation_cfg["time_resolution_minutes"] = 30

ukv_cfg = input_data["nwp"]["ukv"]
ukv_cfg["zarr_path"] = str(UKV_STORE.resolve())
ukv_cfg["interval_start_minutes"] = -120
ukv_cfg["interval_end_minutes"] = 120
ukv_cfg["time_resolution_minutes"] = 60

satellite_cfg = input_data["satellite"]
satellite_cfg["zarr_path"] = str(SATELLITE_STORE.resolve())
# Keep the tested 7-frame satellite request from the example configuration.

solar_cfg = input_data["solar_position"]
solar_cfg["interval_start_minutes"] = -120
solar_cfg["interval_end_minutes"] = 180
solar_cfg["time_resolution_minutes"] = 30

CONFIG_PATH = OUTPUT_DIR / "jan1_week_smoke_configuration.yaml"
with CONFIG_PATH.open("w") as f:
    yaml.safe_dump(data_config, f, sort_keys=False)
print("Wrote run-specific sampler config:", CONFIG_PATH)
print("UKV channels:", len(ukv_cfg["channels"]))
print("Satellite channels:", len(satellite_cfg["channels"]))
print("Generation window: -120 to +180 min; UKV window: -120 to +120 min")


Wrote run-specific sampler config: /Users/mans5211/GitHub-repos/PVNet/Luke/data/to_gitignore/pvnet_training_smoke_test/jan1_week_smoke_configuration.yaml
UKV channels: 17
Satellite channels: 11
Generation window: -120 to +180 min; UKV window: -120 to +120 min


## 3. Build two train samples and one validation sample

In [12]:
import pandas as pd
from pvnet.datamodule import PVNetDataModule

TRAIN_T0S = [pd.Timestamp("2024-01-01 13:00:00"), pd.Timestamp("2024-01-01 14:00:00")]  # naive local-store timestamps are UTC
VAL_T0 = pd.Timestamp("2024-01-04 13:00:00")

datamodule = PVNetDataModule(
    configuration=str(CONFIG_PATH),
    train_periods=[(t.isoformat(sep=" "), t.isoformat(sep=" ")) for t in TRAIN_T0S],
    val_periods=[(VAL_T0.isoformat(sep=" "), VAL_T0.isoformat(sep=" "))],
    batch_size=2,
    num_workers=0,
    prefetch_factor=None,
    persistent_workers=False,
    seed=2727831,
)
datamodule.setup("fit")

print("Train samples:", len(datamodule.train_dataset))
print("Validation samples:", len(datamodule.val_dataset))
if len(datamodule.train_dataset) != 2 or len(datamodule.val_dataset) != 1:
    raise ValueError(
        "Expected two training samples and one validation sample. Check the printed periods and valid data coverage."
    )
print("Train t0s:", datamodule.train_dataset.valid_t0_times, "UTC")
print("Validation t0:", datamodule.val_dataset.dataset.valid_t0_times[0], "UTC")


Train samples: 2
Validation samples: 1
Train t0s: DatetimeIndex(['2024-01-01 13:00:00', '2024-01-01 14:00:00'], dtype='datetime64[ns]', freq=None) UTC
Validation t0: 2024-01-04 13:00:00 UTC


## 4. Check the batches against the model config

In [13]:
from hydra.utils import instantiate
from omegaconf import OmegaConf
from pvnet.utils import validate_batch_against_config

model_config = OmegaConf.load(MODEL_CONFIG_PATH)
# The checked-in example mapping is only illustrative. This run contains GSP 199 alone.
model_config.model.location_id_mapping = {199: 0}
lightning_model = instantiate(model_config)

train_batch = next(iter(datamodule.train_dataloader()))
val_batch = next(iter(datamodule.val_dataloader()))
validate_batch_against_config(train_batch, lightning_model.model)
validate_batch_against_config(val_batch, lightning_model.model)

print("Train batch shapes:")
print("  satellite:", tuple(train_batch["satellite_actual"].shape))
print("  UKV:", tuple(train_batch["nwp"]["ukv"]["nwp"].shape))
print("  generation:", tuple(train_batch["generation"].shape))
print("  solar elevation:", tuple(train_batch["solar_elevation"].shape))
print("Batch shape validation passed for train and validation.")


Train batch shapes:
  satellite: (2, 7, 11, 24, 24)
  UKV: (2, 5, 17, 24, 24)
  generation: (2, 11)
  solar elevation: (2, 11)
Batch shape validation passed for train and validation.


## 5. Run one CPU training step and one validation step

This uses PVNet’s actual Lightning module and optimizer, but disables loggers, checkpoints and progress bars. The single training batch contains two samples so the model’s BatchNorm layers can run; validation uses one sample in evaluation mode. The losses only show that training and validation execute, not forecast quality.


In [14]:
import torch
from lightning.pytorch import Trainer

torch.set_num_threads(2)
trainer = Trainer(
    accelerator="cpu",
    devices=1,
    max_epochs=1,
    limit_train_batches=1,
    limit_val_batches=1,
    num_sanity_val_steps=0,
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
    log_every_n_steps=1,
    default_root_dir=str(OUTPUT_DIR),
)
trainer.fit(
    lightning_model,
    train_dataloaders=datamodule.train_dataloader(),
    val_dataloaders=datamodule.val_dataloader(),
)

print("Global optimizer steps:", trainer.global_step)
print("Epochs completed:", trainer.current_epoch)
print("Metrics:")
for key, value in trainer.callback_metrics.items():
    if hasattr(value, "item"):
        value = value.item()
    print(f"  {key}: {value}")
if trainer.global_step != 1:
    raise RuntimeError(f"Expected one optimizer step, got {trainer.global_step}.")
print("Smoke test passed: one training update and one validation pass completed.")


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/Users/mans5211/GitHub-repos/PVNet/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1)` was configured so 1 batch per epoch will be used.
`Trainer(limit_val_batches=1)` was configured so 1 batch will be used.
/Users/mans5211/GitHub-repos/PVNet/.venv/lib/python3.12/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/mans5211/GitHub-repos/PVNet/.venv/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/data_co

Global optimizer steps: 1
Epochs completed: 1
Metrics:
  quantile_loss/train: 3.235900640487671
  quantile_loss/train_step: 3.235900640487671
  MSE/train: 23.694189071655273
  MSE/train_step: 23.694189071655273
  MAE/train: 3.079068422317505
  MAE/train_step: 3.079068422317505
  quantile_loss/val: 12.22529411315918
  MSE/val: 425.6257629394531
  MAE/val: 16.024045944213867
  val_fraction_below/fraction_below_0.10_quantile: 0.5
  val_fraction_below/fraction_below_0.50_quantile: 1.0
  val_fraction_below/fraction_below_0.90_quantile: 0.5
  val_step_MAE/step_000: 33.1905517578125
  val_step_MAE/step_001: 30.272689819335938
  val_step_MAE/step_002: 13.28113842010498
  val_step_MAE/step_003: 0.23187464475631714
  val_step_MAE/step_004: 18.95303726196289
  val_step_MAE/step_005: 0.21499323844909668
  val_step_MSE/step_000: 1101.6126708984375
  val_step_MSE/step_001: 916.4357299804688
  val_step_MSE/step_002: 176.38864135742188
  val_step_MSE/step_003: 0.05376585200428963
  val_step_MSE/step_0

## What this establishes

A pass means the existing January stores can form correctly shaped PVNet batches and the configured model can complete an optimizer update and validation pass. It does **not** establish useful accuracy: the weights were randomly initialized, the run used only one GSP and two training samples plus one validation sample, and the date split is far too small for model assessment.

If the batch validation fails, keep the printed shapes and config values; if the model step fails, keep the traceback. Those will identify whether the remaining issue is data-window alignment, channel shape, or model execution.
